# Pydantic AI with Gemini — typed output in a notebook

Same agent as `agent_app.py`, with the one change a notebook forces: **`run_sync()` does not work
here.** The kernel is already running an asyncio event loop, so this notebook uses top-level
`await` instead.

Needs `GEMINI_API_KEY` in a `.env` file beside this notebook.

In [1]:
from dotenv import load_dotenv

load_dotenv()  # before the agent is built — the provider reads the key from the environment

from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry


class DatabaseQuery(BaseModel):
    sql_query: str = Field(description="Valid PostgreSQL query")
    explanation: str = Field(description="Brief breakdown of what the query does")


agent = Agent(
    "google:gemini-flash-lite-latest",
    output_type=DatabaseQuery,
    instructions=(
        "You write PostgreSQL against a schema with users(id, name) and "
        "orders(user_id, total_amount, created_at). Answer with the query itself, "
        "never with a schema-inspection query."
    ),
    retries=3,
)
print("agent ready")

agent ready


## The trap: `run_sync()` in a notebook

Worth running once so the error is familiar. This cell is *supposed* to fail.

In [2]:
try:
    agent.run_sync("Select all users.")
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: This event loop is already running


## The fix: top-level `await`

`agent.run(...)` is the coroutine `run_sync` wraps. IPython allows `await` at cell level, so the
notebook's existing loop runs it.

In [3]:
result = await agent.run("Find the top 5 users by total spend in 2026")
print(result.output.sql_query)

SELECT u.id, u.name, SUM(o.total_amount) AS total_spend
FROM users u
JOIN orders o ON u.id = o.user_id
WHERE EXTRACT(YEAR FROM o.created_at) = 2026
GROUP BY u.id, u.name
ORDER BY total_spend DESC
LIMIT 5;


The return value is a real `DatabaseQuery`, not a string — the notebook renders it as the
model it is.

In [4]:
result.output

DatabaseQuery(sql_query='SELECT u.id, u.name, SUM(o.total_amount) AS total_spend\nFROM users u\nJOIN orders o ON u.id = o.user_id\nWHERE EXTRACT(YEAR FROM o.created_at) = 2026\nGROUP BY u.id, u.name\nORDER BY total_spend DESC\nLIMIT 5;', explanation='Find the top 5 users by total spend in 2026 by joining users and orders, filtering by the year 2026, grouping by user, and ordering by total spend in descending order.')

## The validator, and the retry it causes

The schema cannot express "this query has a LIMIT". A validator can, and raising `ModelRetry`
sends the complaint back to the model as a new turn.

In [5]:
attempts = []


@agent.output_validator
def must_be_capped(out: DatabaseQuery) -> DatabaseQuery:
    attempts.append(out.sql_query)
    if "limit" not in out.sql_query.lower():
        raise ModelRetry("The query must include an explicit LIMIT clause. Add one.")
    return out


capped = await agent.run("Show every user's total spend in 2026, ordered by spend.")

for i, a in enumerate(attempts, 1):
    print(f"attempt {i}: {'LIMIT' if 'limit' in a.lower() else 'NO LIMIT'}")
print("requests billed:", capped.usage.requests)

attempt 1: NO LIMIT
attempt 2: LIMIT
requests billed: 2


`requests` is the honest cost of the retry: one prompt, more than one round trip.